In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

# --- File locations ---
# Easiest setup: keep this notebook, Overall Grades.xlsx, and PFF QB.csv in the SAME folder.
BASE_DIR = Path.cwd()

OVERALL_GRADES_FILE = BASE_DIR / "Overall Grades.csv"
QB_GRADES_FILE = BASE_DIR / "PFF QB.csv"
OUTPUT_DIR = BASE_DIR.parent / "PickleFiles"   # matches your repo's existing output location

OUTPUT_DIR.mkdir(exist_ok=True)

SEASON = 2025  # change if your fantasy site expects a different season label

print("Reading:")
print(" -", OVERALL_GRADES_FILE)
print(" -", QB_GRADES_FILE)
print("Saving pickles to:")
print(" -", OUTPUT_DIR)

Reading:
 - c:\Users\shett\Downloads\StepByStepToolKit-main\StepByStepToolKit-main\Models\approximate value data\Overall Grades.csv
 - c:\Users\shett\Downloads\StepByStepToolKit-main\StepByStepToolKit-main\Models\approximate value data\PFF QB.csv
Saving pickles to:
 - c:\Users\shett\Downloads\StepByStepToolKit-main\StepByStepToolKit-main\Models\PickleFiles


In [11]:
# --- Team name / abbreviation standardization ---
TEAM_NAME_TO_ABBR = {
    "Arizona Cardinals": "ARI",
    "Atlanta Falcons": "ATL",
    "Baltimore Ravens": "BAL",
    "Buffalo Bills": "BUF",
    "Carolina Panthers": "CAR",
    "Chicago Bears": "CHI",
    "Cincinnati Bengals": "CIN",
    "Cleveland Browns": "CLE",
    "Dallas Cowboys": "DAL",
    "Denver Broncos": "DEN",
    "Detroit Lions": "DET",
    "Green Bay Packers": "GB",
    "Houston Texans": "HOU",
    "Indianapolis Colts": "IND",
    "Jacksonville Jaguars": "JAX",
    "Kansas City Chiefs": "KC",
    "Las Vegas Raiders": "LV",
    "Los Angeles Chargers": "LAC",
    "Los Angeles Rams": "LAR",
    "Miami Dolphins": "MIA",
    "Minnesota Vikings": "MIN",
    "New England Patriots": "NE",
    "New Orleans Saints": "NO",
    "New York Giants": "NYG",
    "New York Jets": "NYJ",
    "Philadelphia Eagles": "PHI",
    "Pittsburgh Steelers": "PIT",
    "San Francisco 49ers": "SF",
    "Seattle Seahawks": "SEA",
    "Tampa Bay Buccaneers": "TB",
    "Tennessee Titans": "TEN",
    "Washington Commanders": "WAS",
}

PFF_QB_ABBR_TO_SITE_ABBR = {
    "ARZ": "ARI",
    "ATL": "ATL",
    "BLT": "BAL",
    "BUF": "BUF",
    "CAR": "CAR",
    "CHI": "CHI",
    "CIN": "CIN",
    "CLV": "CLE",
    "DAL": "DAL",
    "DEN": "DEN",
    "DET": "DET",
    "GB": "GB",
    "HST": "HOU",
    "IND": "IND",
    "JAX": "JAX",
    "KC": "KC",
    "LA": "LAR",
    "LAC": "LAC",
    "LV": "LV",
    "MIA": "MIA",
    "MIN": "MIN",
    "NE": "NE",
    "NO": "NO",
    "NYG": "NYG",
    "NYJ": "NYJ",
    "PHI": "PHI",
    "PIT": "PIT",
    "SEA": "SEA",
    "SF": "SF",
    "TB": "TB",
    "TEN": "TEN",
    "WAS": "WAS",
}

In [14]:

# --- Read team-level grades from CSV ---
team_df = pd.read_csv(OVERALL_GRADES_FILE).copy()

team_df["team"] = team_df["Team"].map(TEAM_NAME_TO_ABBR)
missing_team_names = team_df.loc[team_df["team"].isna(), "Team"].unique().tolist()
if missing_team_names:
    raise ValueError(f"Unmapped team names in Excel: {missing_team_names}")

# Map existing values into the format your site already expects
team_df["pass_oline"] = team_df["Pass Blocking PFF Grade"]
team_df["run_oline"] = team_df["Run Blocking PFF Grade"]
team_df["rb"] = team_df["Running PFF Grade"]
team_df["wrte"] = team_df["Receiving PFF Grade"]
team_df["dst"] = team_df["Defense PFF Grade"]
team_df["season"] = SEASON

team_base = team_df[["team", "pass_oline", "run_oline", "rb", "wrte", "dst", "season"]].copy()
team_base.head()


,team,pass_oline,run_oline,rb,wrte,dst,season
0,ARI,60.5,55.3,75.6,74.2,50.7,2025
1,ATL,68.2,71.0,84.8,74.1,65.7,2025
2,BAL,62.8,71.9,84.8,73.8,69.8,2025
3,BUF,72.3,75.1,90.4,74.9,60.8,2025
4,CAR,67.4,75.7,76.7,66.7,60.4,2025


In [15]:
# --- Read QB grades from CSV ---
qb_df = pd.read_csv(QB_GRADES_FILE).copy()

qb_df["team"] = qb_df["Team Abbreviation"].map(PFF_QB_ABBR_TO_SITE_ABBR)
missing_qb_abbr = qb_df.loc[qb_df["team"].isna(), "Team Abbreviation"].unique().tolist()
if missing_qb_abbr:
    raise ValueError(f"Unmapped QB team abbreviations in CSV: {missing_qb_abbr}")

# If multiple QBs exist for a team, use a snaps-played weighted average
qb_team = (
    qb_df.assign(weighted_grade=qb_df["Overall Grade"] * qb_df["Snaps Played"])
         .groupby("team", as_index=False)
         .agg(
             weighted_grade_sum=("weighted_grade", "sum"),
             snaps_sum=("Snaps Played", "sum"),
             qb_mean=("Overall Grade", "mean")
         )
)

qb_team["qb"] = np.where(
    qb_team["snaps_sum"] > 0,
    qb_team["weighted_grade_sum"] / qb_team["snaps_sum"],
    qb_team["qb_mean"]
)

qb_team = qb_team[["team", "qb"]]
qb_team.head()

,team,qb
0,ARI,70.600000
1,ATL,60.934913
2,BAL,74.000000
3,BUF,90.500000
4,CAR,71.000000


In [17]:
# --- Merge to final site format ---
AVgrades = team_base.merge(qb_team, on="team", how="left")

# Fallback: if a team has no QB row in the CSV, use team pass grade from the Excel file
pass_grade_fallback = team_df[["team", "Pass PFF Grade"]].rename(columns={"Pass PFF Grade": "qb_fallback"})
AVgrades = AVgrades.merge(pass_grade_fallback, on="team", how="left")
AVgrades["qb"] = AVgrades["qb"].fillna(AVgrades["qb_fallback"])
AVgrades = AVgrades.drop(columns=["qb_fallback"])

# Final column order expected by the rest of the site
AVgrades = AVgrades[["team", "pass_oline", "run_oline", "qb", "rb", "wrte", "dst", "season"]].sort_values(["season", "team"]).reset_index(drop=True)

print(f"AVgrades shape: {AVgrades.shape}")
print(f"Missing QBs after fallback: {AVgrades['qb'].isna().sum()}")
AVgrades.head(10)


AVgrades shape: (32, 8)
Missing QBs after fallback: 0


,team,pass_oline,run_oline,qb,rb,wrte,dst,season
0,ARI,60.5,55.3,70.600000,75.6,74.2,50.7,2025
1,ATL,68.2,71.0,60.934913,84.8,74.1,65.7,2025
2,BAL,62.8,71.9,74.000000,84.8,73.8,69.8,2025
3,BUF,72.3,75.1,90.500000,90.4,74.9,60.8,2025
4,CAR,67.4,75.7,71.000000,76.7,66.7,60.4,2025
5,CHI,73.1,75.3,76.900000,90.2,75.2,61.6,2025
6,CIN,61.1,55.8,77.748415,77.1,80.2,52.6,2025
7,CLE,49.7,55.4,46.378500,69.5,60.9,84.5,2025
8,DAL,53.7,71.4,86.900000,76.2,81.7,52.8,2025
9,DEN,79.0,72.0,77.100000,77.9,67.6,80.2,2025


In [18]:
# --- Save outputs with the same filenames the repo already uses ---
AVgrades.to_pickle(OUTPUT_DIR / "AVgrades.pkl")

AVbyPositionGroup = AVgrades[["team", "pass_oline", "run_oline", "qb", "rb", "wrte", "dst", "season"]].copy()
AVbyPositionGroup.to_pickle(OUTPUT_DIR / "AVbyPositionGroup.pkl")

print("Saved:")
print(" -", OUTPUT_DIR / "AVgrades.pkl")
print(" -", OUTPUT_DIR / "AVbyPositionGroup.pkl")


Saved:
 - c:\Users\shett\Downloads\StepByStepToolKit-main\StepByStepToolKit-main\Models\PickleFiles\AVgrades.pkl
 - c:\Users\shett\Downloads\StepByStepToolKit-main\StepByStepToolKit-main\Models\PickleFiles\AVbyPositionGroup.pkl
